In [ ]:
import os
import pandas as pd
import numpy as np
import glob
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import joblib

def create_position_dataset(data, missing_position):
    """
    Creates a dataset for training/testing with a specific position missing.
    """
    # Copy data to avoid modifying original
    position_data = data.copy()
    
    # Define column groups
    home_players = [f'home_{i}' for i in range(5)]
    away_players = [f'away_{i}' for i in range(5)]
    model_features = ['season', 'home_team', 'away_team', 'starting_min']
    
    # Remove the missing position from input features
    input_home_players = [p for p in home_players if p != missing_position]
    input_features = model_features + input_home_players + away_players
    target_column = missing_position
    
    # Store original team names and game ids
    original_home_teams = position_data['home_team'].copy()
    original_away_teams = position_data['away_team'].copy()
    game_ids = position_data['game'].copy()
    
    # Select features and target
    X = position_data[input_features].copy()
    y = position_data[target_column].copy()
    
    return X, y, input_features, target_column, game_ids, original_home_teams, original_away_teams

def train_position_model(X, y, input_features, target_column, game_ids, original_home_teams, original_away_teams):
    """
    Trains a model for a specific missing position and restricts predictions to players from the home team.
    """
    # Store original data
    X_orig = X.copy()
    
    # Encode categorical variables (teams and players)
    categorical_columns = ['home_team', 'away_team'] + [col for col in X.columns if 'home_' in col or 'away_' in col]
    label_encoders = {}
    for col in categorical_columns:
        label_encoders[col] = LabelEncoder()
        X[col] = label_encoders[col].fit_transform(X[col])
    
    # Encode target player
    label_encoders[target_column] = LabelEncoder()
    y = label_encoders[target_column].fit_transform(y)
    
    # Split the data with reference data
    X_train, X_test, y_train, y_test, game_train, game_test, home_team_train, home_team_test, away_team_train, away_team_test = train_test_split(
        X, y, game_ids, original_home_teams, original_away_teams, test_size=0.2, random_state=42
    )
    
    # Scale only the starting_min feature
    scaler = StandardScaler()
    X_train['starting_min'] = scaler.fit_transform(X_train[['starting_min']])
    X_test['starting_min'] = scaler.transform(X_test[['starting_min']])
    
    # Initialize and train the Random Forest model
    rf_model = RandomForestClassifier(
        n_estimators=200,
        max_depth=20,
        min_samples_split=4,
        min_samples_leaf=1,
        max_features='sqrt',
        class_weight='balanced',
        n_jobs=-1,
        random_state=42
    )
    rf_model.fit(X_train, y_train)
    
    # Build allowed players mapping from training data:
    # For each home team (using original team names), record the player from the target column.
    allowed_players_by_team = {}
    for team, encoded_player in zip(home_team_train, y_train):
        player_name = label_encoders[target_column].inverse_transform([encoded_player])[0]
        if team not in allowed_players_by_team:
            allowed_players_by_team[team] = set()
        allowed_players_by_team[team].add(player_name)
    
    # Use the model's classes (the classes the model was trained on)
    model_classes = rf_model.classes_  # encoded labels present in training
    # Convert these encoded labels to player names in the same order as rf_probabilities columns.
    model_player_names = label_encoders[target_column].inverse_transform(model_classes)
    
    # Get predicted probabilities on the test set
    rf_probabilities = rf_model.predict_proba(X_test)
    
    # For each test sample, restrict predictions to players from the corresponding home team
    filtered_predictions = []
    for i, prob in enumerate(rf_probabilities):
        # Get the original home team name from the test split
        team = home_team_test.iloc[i] if hasattr(home_team_test, 'iloc') else home_team_test[i]
        allowed = allowed_players_by_team.get(team, None)
        if allowed:
            # Get indices of model_player_names corresponding to allowed players
            allowed_indices = [j for j, player in enumerate(model_player_names) if player in allowed]
            if allowed_indices:
                allowed_probs = prob[allowed_indices]
                best_index = allowed_indices[np.argmax(allowed_probs)]
            else:
                best_index = np.argmax(prob)
        else:
            best_index = np.argmax(prob)
        filtered_predictions.append(best_index)
    filtered_predictions = np.array(filtered_predictions)
    
    # Calculate accuracy using the filtered predictions
    filtered_accuracy = accuracy_score(y_test, filtered_predictions)
    
    # Create a results DataFrame
    results = pd.DataFrame()
    results['game'] = game_test
    results['season'] = X_test['season']
    results['home_team'] = home_team_test  # Original team names
    results['away_team'] = away_team_test  # Original team names
    results['actual_player'] = label_encoders[target_column].inverse_transform(y_test)
    results['predicted_player'] = label_encoders[target_column].inverse_transform(filtered_predictions)
    
    # Calculate feature importances
    feature_importances = pd.DataFrame(
        rf_model.feature_importances_,
        index=input_features,
        columns=['importance']
    ).sort_values('importance', ascending=False)
    
    print(f"Filtered Model Accuracy for {target_column}: {filtered_accuracy:.4f}")
    
    return rf_model, label_encoders, scaler, results, feature_importances, filtered_accuracy

def save_all_results(results_all, models, output_filename='nba_predictions_results_all.xlsx'):
    """
    Saves all results into one Excel file with separate sheets for each missing position's predictions 
    and feature importances.
    """
    with pd.ExcelWriter(output_filename, mode='w') as writer:
        for position, results in results_all.items():
            # Save predictions
            sheet_name_pred = f'{position}_predictions'
            results.to_excel(writer, sheet_name=sheet_name_pred, index=False)
            # Save feature importances; models[position]['feature_importances'] holds the dataframe
            sheet_name_imp = f'{position}_feature_importance'
            models[position]['feature_importances'].to_excel(writer, sheet_name=sheet_name_imp)
    print(f"All results successfully saved to {output_filename}")

# Define the directory and file pattern for matchup files
matchup_files = glob.glob("Datasets/matchups-*.csv")

data_frames = []
for file in matchup_files:
    try:
        df = pd.read_csv(file)
        data_frames.append(df)
    except Exception as e:
        print(f"Error loading {file}: {e}")

# Combine all dataframes, drop NaNs and duplicates
combined_data = pd.concat(data_frames, ignore_index=True).dropna()
combined_data.drop_duplicates(inplace=True)

# Filter for winning home team samples
combined_data = combined_data[combined_data['outcome'] == 1]

print(f"\nNumber of winning home team samples: {len(combined_data)}")
print(f"Seasons covered: {combined_data['season'].unique().tolist()}")

# Train models for each possible missing home position
models = {}
results_all = {}
for position in [f'home_{i}' for i in range(5)]:
    print(f"\nTraining model for missing position: {position}")
    
    X, y, input_features, target_column, game_ids, original_home_teams, original_away_teams = create_position_dataset(combined_data, position)
    model, encoders, scaler, results, importances, accuracy = train_position_model(
        X, y, input_features, target_column, game_ids, original_home_teams, original_away_teams
    )
    
    models[position] = {
        'model': model,
        'encoders': encoders,
        'scaler': scaler,
        'feature_importances': importances,
        'accuracy': accuracy
    }
    results_all[position] = results
    
    print(f"Model Accuracy for {position}: {accuracy:.4f}")

# Save all results into one Excel file
save_all_results(results_all, models)
print("\nAll models trained and results saved successfully!")


Number of winning home team samples: 88175
Seasons covered: [2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015]

Training model for missing position: home_0
Fitting 2 folds for each of 10 candidates, totalling 20 fits


C:\Users\paras\anaconda3\Lib\site-packages\sklearn\model_selection\_split.py:776: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=2.
  warnings.warn(
